# 05_logistic_regression_model

In [ ]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path

from features import build_features

sys.path.append(str(Path.cwd().parent / "src"))

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from models import train_baseline_model
from models import evaluate_model
from load_data import load_model_data
from models import train_baseline_model, prepare_data, evaluate_model


df = load_model_data()
df = build_features(df)

from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
)

df.head()


,timestamp,pool_price,demand,IMPORT_BC,IMPORT_MT,IMPORT_SK,EXPORT_BC,EXPORT_MT,EXPORT_SK,calgary_temperature_2m,...,abs_demand_ramp,abs_net_load_ramp,abs_gen_ramp,demand_ramp_pct,net_load_ramp_pct,wind_ramp,solar_ramp,wind_instability_24h,solar_instability_24h,ramp_stress
0,2020-01-01 00:00:00,30.24,9467,0,80,0,0,0,0,0.55,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-01-01 01:00:00,29.43,9361,0,125,0,0,0,0,-0.20,...,106.0,10.664491,127.208351,-0.011324,-0.001139,-95.335509,0.0,NaN,NaN,0.089064
2,2020-01-01 02:00:00,29.66,9286,0,125,0,0,0,0,-0.50,...,75.0,11.999348,89.532789,-0.008077,-0.001292,-63.000653,0.0,NaN,NaN,0.089365
3,2020-01-01 03:00:00,28.19,9209,0,125,0,0,0,0,-1.55,...,77.0,57.806784,64.027290,-0.008361,-0.006277,-19.193216,0.0,NaN,NaN,0.476560
4,2020-01-01 04:00:00,31.17,9195,0,125,0,0,0,0,-2.10,...,14.0,122.786104,25.196383,-0.001523,0.013354,-136.786104,0.0,NaN,NaN,0.926713


In [8]:
df['scarcity_event'] = (df['pool_price'] > 300).astype(int)

print('Scarcity rate:', df['scarcity_event'].mean().round(4))

Scarcity rate: 0.0705


In [9]:
base_features = [
    "net_load",
    "renewable_share",
    "gen_wind",
    "reserve_proxy_pct",
    "total_imports",
    "wind_instability_24h"
]

In [10]:
interaction_features = [
    'ramp_x_netload',
    'ramp_x_reserve'
]

In [12]:
X_train, X_test, y_train, y_test, feature_names = prepare_data(df, base_features)

log_reg = LogisticRegression(max_iter=1000)

log_reg.fit(X_train, y_train)

evaluate_model(log_reg, X_test, y_test)


=== MODEL PERFORMANCE ===
AUC: 0.854
Precision: 0.618
Recall: 0.091

Confusion Matrix:
[[9054   39]
 [ 627   63]]


(0.8538557291243303,
 0.6176470588235294,
 0.09130434782608696,
 array([[9054,   39],
        [ 627,   63]]))

In [13]:
baseline_auc, baseline_precision, baseline_recall, baseline_cm = evaluate_model(
    log_reg, X_test, y_test
)


=== MODEL PERFORMANCE ===
AUC: 0.854
Precision: 0.618
Recall: 0.091

Confusion Matrix:
[[9054   39]
 [ 627   63]]


In [14]:
coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": log_reg.coef_[0]
}).sort_values("coefficient", ascending=False)

print(coef_df)

                feature  coefficient
0              net_load     1.482851
3     reserve_proxy_pct     0.518946
1       renewable_share     0.335426
5  wind_instability_24h     0.076471
4         total_imports     0.012281
2              gen_wind    -1.425169


In [15]:
print("\nTop positive drivers of scarcity (increase odds):")
print(coef_df.head(5))

print("\nTop negative drivers of scarcity (decrease odds):")
print(coef_df.tail(5))


Top positive drivers of scarcity (increase odds):
                feature  coefficient
0              net_load     1.482851
3     reserve_proxy_pct     0.518946
1       renewable_share     0.335426
5  wind_instability_24h     0.076471
4         total_imports     0.012281

Top negative drivers of scarcity (decrease odds):
                feature  coefficient
3     reserve_proxy_pct     0.518946
1       renewable_share     0.335426
5  wind_instability_24h     0.076471
4         total_imports     0.012281
2              gen_wind    -1.425169


#### Part 2:

In [17]:
interaction_model_features = base_features + interaction_features

In [25]:
X_train, X_test, y_train, y_test, feature_names = prepare_data(df, interaction_model_features)

log_reg_int = LogisticRegression(max_iter=2000)
log_reg_int.fit(X_train, y_train)

KeyError: "['ramp_x_netload', 'ramp_x_reserve'] not in index"

In [20]:
interaction_auc, interaction_precision, interaction_recall, interaction_cm = evaluate_model(
    log_reg_int, X_test, y_test
)


=== MODEL PERFORMANCE ===
AUC: 0.854
Precision: 0.618
Recall: 0.091

Confusion Matrix:
[[9054   39]
 [ 627   63]]


In [21]:
print("\n=== MODEL COMPARISON ===")

print(f"Stress Score (Notebook 04): 0.842")
print(f"Logistic Regression (baseline): {baseline_auc:.3f}")
print(f"Logistic Regression (with interactions): {interaction_auc:.3f}")


=== MODEL COMPARISON ===
Stress Score (Notebook 04): 0.842
Logistic Regression (baseline): 0.854
Logistic Regression (with interactions): 0.854


#### Part 3:

In [22]:
coef_df_int = pd.DataFrame({
    "feature": feature_names,
    "coefficient": log_reg_int.coef_[0]
}).sort_values("coefficient", ascending=False)

print(coef_df_int)

                feature  coefficient
0              net_load     1.482851
3     reserve_proxy_pct     0.518946
1       renewable_share     0.335426
5  wind_instability_24h     0.076471
4         total_imports     0.012281
2              gen_wind    -1.425169


In [23]:
print("\n=== INTERPRETATION ===")

print("\nWhat increases scarcity odds?")
print(coef_df_int.head(5))

print("\nWhat decreases scarcity odds?")
print(coef_df_int.tail(5))


=== INTERPRETATION ===

What increases scarcity odds?
                feature  coefficient
0              net_load     1.482851
3     reserve_proxy_pct     0.518946
1       renewable_share     0.335426
5  wind_instability_24h     0.076471
4         total_imports     0.012281

What decreases scarcity odds?
                feature  coefficient
3     reserve_proxy_pct     0.518946
1       renewable_share     0.335426
5  wind_instability_24h     0.076471
4         total_imports     0.012281
2              gen_wind    -1.425169


In [24]:
interaction_only = coef_df_int[
    coef_df_int["feature"].isin(interaction_features)
]

print("\nInteraction term coefficients:")
print(interaction_only)


Interaction term coefficients:
Empty DataFrame
Columns: [feature, coefficient]
Index: []
